# Notebook 08 - Synthetic Signal Analysis

This notebook analyzes invented values from anonymous signal channels. Results are illustrative and are not sourced from an organization or operating asset.

## Analysis step 2

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib

from sklearn.impute import SimpleImputer
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

HERE = Path.cwd().resolve()
ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / 'data' / 'processed' / 'features_labels.csv').is_file():
        ROOT = candidate
        break
assert ROOT is not None, 'cannot find the project root'
sys.path.insert(0, str(ROOT))

from src.scoring import preprocess_sensors, compute_features, score_new, assert_bundle_complete

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLE_DIR = ROOT / 'reports' / 'tables'
FIG_DIR = ROOT / 'reports' / 'figures'
MODEL_DIR = ROOT / 'models'
for d in (TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

cfg = json.load(open(PROCESSED / 'label_config.json', encoding='utf-8'))
CORE = cfg['core_sensors']
MAIN_H = int(cfg['main_horizon'])
FAILURE_DATE = pd.Timestamp(cfg['failure_date_used']).normalize()

clean = pd.read_csv(INTERIM / 'channel_clean.csv', index_col=0, parse_dates=True)
scaled = pd.read_csv(INTERIM / 'channel_scaled.csv', index_col=0, parse_dates=True)
table = pd.read_csv(PROCESSED / 'features_labels.csv', index_col=0, parse_dates=True)
anom_cal = pd.read_csv(TABLE_DIR / '06_anomaly_calibration_summary.csv')

assert clean.index.equals(scaled.index) and clean.index.equals(table.index)
print('project root:', ROOT)
print('failure date used:', FAILURE_DATE.date(), '| provisional:', cfg['failure_date_provisional'])
print('feature rows:', table.shape, '| core sensors:', len(CORE))

state_cols = ['is_shutdown', 'is_restart', 'is_suspect', 'is_baseline', 'core_complete', 'segment_id']
state = clean[state_cols].copy()
for c in state_cols[:-1]:
    state[c] = state[c].astype(bool)
seg = pd.to_numeric(state['segment_id'], errors='coerce')
X = scaled[CORE].astype(float)

valid = (~state['is_shutdown'] & ~state['is_restart'] & ~state['is_suspect']
         & state['core_complete'] & X.notna().all(axis=1))
baseline = valid & state['is_baseline'] & (clean.index.year == 2032)
baseline_dates = clean.index[baseline]
cut = int(np.floor(len(baseline_dates) * 0.70))
fit_end = baseline_dates[cut - 1]
fit_mask = baseline & (clean.index <= fit_end)
cal_mask = baseline & (clean.index > fit_end)
ev_start = clean.index[cal_mask][-1].normalize() + pd.Timedelta(days=1)

use = table['usable'].astype(bool)
lvalid = table['label_valid'].astype(bool)
split = table['split']
print('usable rows:', int(use.sum()), '| eligible 2033 operating days:', int((valid & (clean.index.year == 2033)).sum()))


## Analysis step 3

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
sys.path.insert(0, str(ROOT))
from src.channel_io import read_raw_sheet, HEADER_ROWS, DATE_COL, find_raw_file

raw_found = False
try:
    raw = read_raw_sheet(find_raw_file(ROOT / 'data' / 'raw'))
    raw_idx = pd.to_datetime(raw.iloc[HEADER_ROWS:, DATE_COL], errors='coerce')
    raw_found = True
except Exception as exc:  # raw workbook absent (fresh clone without the private file)
    print('raw workbook not present:', exc)

if raw_found:
    print('raw rows:', len(raw_idx), '| span:', raw_idx.min().date(), '->', raw_idx.max().date())
    print('rows by year:', raw_idx.dt.year.value_counts().sort_index().to_dict())
    print('any 2026 rows:', bool((raw_idx.dt.year == 2026).any()))

segs = []
seg_ids = pd.to_numeric(clean['segment_id'], errors='coerce')
for sid in seg_ids.dropna().unique():
    g = clean.loc[seg_ids.index[seg_ids.eq(sid)]]
    segs.append({'segment_id': int(sid), 'start': g.index[0].date(), 'end': g.index[-1].date(),
                 'operating_days': len(g)})
segments = pd.DataFrame(segs)

shutdowns = pd.read_csv(INTERIM / 'state_segments.csv')
print()
print('operating segments:')
print(segments.to_string(index=False))
print()
print('shutdown events (signal rule):')
print(shutdowns[['start', 'end', 'days', 'rule']].to_string(index=False))

avail_rows = [
    {'period': 'raw records', 'start': '2032-01-01', 'end': '2033-12-31',
     'days': 731, 'role': 'only data present', 'note': 'no 2026 rows exist in the workbook'},
    {'period': 'healthy-reference fit (2032 baseline split)', 'start': str(baseline_dates[0].date()),
     'end': str(baseline_dates[cut - 1].date()), 'days': int(fit_mask.sum()),
     'role': 'anomaly-detector fit (notebook 06)', 'note': '2032 operating days with complete core'},
    {'period': 'anomaly calibration block', 'start': str(baseline_dates[cut].date()),
     'end': str(baseline_dates[-1].date()), 'days': int(cal_mask.sum()),
     'role': 'threshold calibration (notebook 06)', 'note': 'held-out 2032 healthy days'},
    {'period': 'supervised train', 'start': clean.index.min().date(), 'end': cfg['train_end'],
     'days': int((use & (split == 'train')).sum()),
     'role': 'training rows (usable)', 'note': '0 positives at every horizon'},
    {'period': 'supervised validation', 'start': '2033-01-01', 'end': cfg['val_end'],
     'days': int((use & (split == 'validation')).sum()),
     'role': 'validation rows (usable)', 'note': '0 positives at every horizon'},
    {'period': 'supervised test', 'start': '2033-07-01', 'end': '2033-12-31',
     'days': int((use & (split == 'test')).sum()),
     'role': 'test rows (usable)', 'note': 'only split that contains positives'},
    {'period': 'failure window (last 30 days)', 'start': str((FAILURE_DATE - pd.Timedelta(days=30)).date()),
     'end': str(FAILURE_DATE.date()), 'days': int(((use & lvalid)
             & (table.index >= (FAILURE_DATE - pd.Timedelta(days=30)))).sum()),
     'role': 'single positive block', 'note': '31 usable positives, one contiguous block'},
    {'period': 'evaluation window (notebook 06)', 'start': str(ev_start.date()), 'end': '2033-12-31',
     'days': int((valid & (clean.index.year == 2033)).sum()),
     'role': 'anomaly scoring window', 'note': 'eligible operating days'},
    {'period': '2026 healthy period', 'start': 'n/a', 'end': 'n/a', 'days': 0,
     'role': 'proposed 2032+2026 calibration split', 'note': 'IMPOSSIBLE: the workbook has no 2026 rows'},
]
avail = pd.DataFrame(avail_rows)
avail.to_csv(TABLE_DIR / '08_data_availability.csv', index=False)
print()
print('Availability summary saved to reports/tables/08_data_availability.csv')
avail


## Analysis step 4

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
print('label provenance:')
print('  failure_date_used       :', cfg['failure_date_used'], '(provisional =', cfg['failure_date_provisional'], ')')
print('  estimated_onset         :', cfg.get('estimated_onset'))
print('  horizons                :', cfg['horizons'], '| main:', cfg['main_horizon'])
print('  rul_cap                 :', cfg['rul_cap'])
print()
for hz in cfg['horizons']:
    lab = table[f'label_h{hz}'].astype(int)
    per_split = (use & lvalid & lab.eq(1)).groupby(split).sum()
    print(f'positives h{hz}: train={per_split.loc["train"]}, val={per_split.loc["validation"]}, '
          f'test={per_split.loc["test"]}')
print()
pos = table.index[use & lvalid & table['label_h30'].eq(1)]
if len(pos):
    d = np.diff(pos.normalize().to_numpy().astype('datetime64[D]'))
    n_clusters = int((d > 1).sum()) + 1
    print('main-horizon positives:', len(pos), '| contiguous blocks:', n_clusters,
          '| span:', pos.min().date(), '->', pos.max().date())
print()
print('edge cases handled in label/row selection:')
print('  shutdown days are label-valid?  -> False (excluded)')
print('  restart days usable?            -> False (excluded by notebook 04)')
print('  days after the event usable?    -> False (label_valid = False)')
print('  days before the event are 0s    -> the single provisional event provides all positives')
print('  supervised walk-forward OOF     -> 2 folds trained, 48 skipped, 17 scored rows, all positive')


### Analysis step 5

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 6

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
imputer = SimpleImputer(strategy='median')
X_fit = imputer.fit_transform(X.loc[fit_mask])
X_all = pd.DataFrame(imputer.transform(X), index=X.index, columns=CORE)

lw = LedoitWolf().fit(X_fit)
iso = IsolationForest(n_estimators=300, contamination='auto', random_state=42).fit(X_fit)
pca = PCA(n_components=0.90, svd_solver='full', random_state=42).fit(X_fit)

def pca_error(a):
    return np.mean((a - pca.inverse_transform(pca.transform(a))) ** 2, axis=1)

anom = pd.DataFrame(index=X.index)
for name in ['Ledoit-Wolf distance', 'Isolation Forest', 'PCA reconstruction error']:
    anom[name] = np.nan
v = X_all.loc[valid].to_numpy()
anom.loc[valid, 'Ledoit-Wolf distance'] = np.sqrt(np.maximum(lw.mahalanobis(v), 0))
anom.loc[valid, 'Isolation Forest'] = -iso.score_samples(v)
anom.loc[valid, 'PCA reconstruction error'] = pca_error(v)

thr = anom_cal.set_index(['model', 'calibration_quantile'])['threshold']
SUSTAIN_WINDOW, SUSTAIN_COUNT = 7, 5

def sustained_alarm(raw, segment_id, window=SUSTAIN_WINDOW, count=SUSTAIN_COUNT):
    out = pd.Series(False, index=raw.index)
    for sid in segment_id.dropna().unique():
        idx = segment_id.index[segment_id.eq(sid)]
        s = raw.reindex(idx).fillna(False).astype(int)
        out.loc[idx] = (s.rolling(window, min_periods=window).sum() >= count).to_numpy()
    return out

def episode_starts(sust, segment_id):
    stars = pd.Series(False, index=sust.index)
    for sid in segment_id.dropna().unique():
        idx = segment_id.index[segment_id.eq(sid)]
        sa = sust.loc[idx]
        stars.loc[idx] = sa & ~sa.shift(1, fill_value=False)
    return stars

records = []
sust_registry = {}
for (m, q), th in thr.items():
    raw = (anom[m] > th) & valid
    sust = sustained_alarm(raw, seg)
    sust_registry[(m, q)] = sust
    starts = episode_starts(sust, seg)
    for d in starts.index[starts & (starts.index >= ev_start)]:
        eidx = seg.index[seg.eq(seg.at[d])]
        run = eidx[sust.loc[eidx] & (eidx >= d)]
        end = run[-1] if len(run) else d
        records.append({'model': m, 'quantile': q, 'start': d, 'end': end,
                        'days': (end - d).days + 1, 'segment_id': int(seg.at[d])})

episodes = pd.DataFrame(records).sort_values(['start', 'model', 'quantile'])
episodes.to_csv(TABLE_DIR / '08_alarm_episodes_by_model.csv', index=False)
print('sustained-alarm episodes across all detector/quantile settings:', len(episodes))
print()
print(episodes.groupby(['model', 'quantile']).size().to_string())

union_days = pd.Series(False, index=anom.index)
for sust in sust_registry.values():
    union_days = union_days | (sust & valid & (sust.index >= ev_start))
blk = (union_days != union_days.shift()).cumsum()
union = []
for _, g in union_days[union_days].groupby(blk[union_days]):
    union.append({'start': g.index.min(), 'end': g.index.max(), 'days': len(g)})
u = pd.DataFrame(union).sort_values('start')
print()
print('union episodes (any detector setting active):', len(u))


In [ ]:
fit_med = X_all.loc[fit_mask].median()
horizon_start = FAILURE_DATE - pd.Timedelta(days=MAIN_H)

def describe(ep):
    d0, d1 = ep['start'], ep['end']
    med = X_all.loc[d0:d1].median()
    dev = (med - fit_med)
    top = dev.abs().sort_values(ascending=False).head(4)
    top_str = ', '.join(f'{c}={dev[c]:+.2f}' for c in top.index)
    segid = int(seg.at[d0])
    sd = state.loc[d0:d1]
    restart_in = bool(sd['is_restart'].any())
    shutdown_in = bool(sd['is_shutdown'].any())
    horizon_overlap = bool((d0 <= FAILURE_DATE) and (d1 >= horizon_start))
    terminal = bool(dev['channel_098'] < -5)
    near_restart = None
    for _, r in shutdowns.iterrows():
        re = pd.Timestamp(r['end']).tz_localize(None)
        if re <= d0 <= re + pd.Timedelta(days=15):
            near_restart = (d0 - re).days

    if terminal:
        op = 'terminal window: signal collapsed, signal up (segment ' + str(segid) + ')'
        interp, resolved = ('Only episode concurrent with the configured event; the signal collapse '
                            'is also consistent with a failed/lost transmitter (EDA flagged it).'), False
    elif shutdown_in or restart_in or near_restart is not None:
        op = 'around a shutdown/restart (segment ' + str(segid) + ')'
        interp, resolved = ('Sustained right after system restart; restart transient vs drift '
                            'cannot be separated with this data.'), False
    elif horizon_overlap:
        op = 'drift continuing into the failure window (segment ' + str(segid) + ')'
        interp, resolved = 'General drift; whether it is a failure precursor is unresolved.', False
    else:
        op = 'general 2033 drift (segment ' + str(segid) + ')'
        interp, resolved = 'signal dominates; consistent with the documented 2033 signal drift.', True
    return (op, interp, resolved, top_str, float(dev['channel_098']), float(dev['channel_099_log']))

rows = []
for i, (_, ep) in enumerate(u.iterrows(), 1):
    op, interp, resolved, top_str, signal_dev, c3_dev = describe(ep)
    rows.append({'episode': i, 'start': ep['start'].date(), 'end': ep['end'].date(),
                 'days': ep['days'], 'segment': int(seg.at[ep['start']]),
                 'operating_state': op, 'signal_dev_vs_2032': round(signal_dev, 2),
                 'signal_dev_vs_2032': round(c3_dev, 2),
                 'leading_sensors': top_str,
                 'interpretation': interp, 'resolved': resolved})
union_tab = pd.DataFrame(rows)
union_tab.to_csv(TABLE_DIR / '08_alarm_episodes_union.csv', index=False)
display(union_tab)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
for _, r in union_tab.iterrows():
    left = (pd.Timestamp(r['start']) - ev_start).days
    ax.barh('any detector', r['days'], left=left, height=0.5, color='#E84B4B', alpha=0.85)
    ax.text(left + r['days'] / 2, -0.15, str(r['episode']), ha='center', fontsize=8)
for _, srow in shutdowns.iterrows():
    a = pd.Timestamp(srow['start']).tz_localize(None)
    b = pd.Timestamp(srow['end']).tz_localize(None)
    ax.barh('any detector', (b - a).days + 1, left=(a - ev_start).days, height=0.5,
            color='#AAAAAA', alpha=0.7)
ax.axvspan((horizon_start - ev_start).days, (FAILURE_DATE - ev_start).days, color='#E84B4B', alpha=0.12, lw=0)
ax.set_xlim(0, (FAILURE_DATE - ev_start).days + 5)
ticks = pd.date_range(ev_start, FAILURE_DATE, freq='MS')
ax.set_xticks([(t - ev_start).days for t in ticks])
ax.set_xticklabels([t.strftime('%b %Y') for t in ticks], rotation=30, ha='right', fontsize=8)
ax.set_yticks([])
ax.set_title('Sustained anomaly-alarm union episodes (red), shutdowns (grey), band = last 30 days')
fig.tight_layout()
fig.savefig(FIG_DIR / 'notebook_08_plot_01.png', dpi=130, bbox_inches='tight')
plt.show()
print('numbers mark union episodes 1-8; grey bars are the planned shutdowns.')


### Analysis step 7

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 8

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
bundles = {
    'logreg': joblib.load(MODEL_DIR / '05_logreg_early_warning.joblib'),
    'tree': joblib.load(MODEL_DIR / '05_tree_early_warning.joblib'),
}
for name, b in bundles.items():
    assert_bundle_complete(b)
    steps = [(s[0], type(s[1]).__name__) for s in b['pipeline'].steps]
    print(name, '| features:', len(b['features']), '| horizon:', b['horizon'],
          '| threshold:', b['threshold'], '| sustain_k:', b['sustain_k'])
    print('   pipeline steps:', steps)
    print('   sensor_scaler columns:', len(b['sensor_scaler']['columns']),
          '| fit rows:', b['sensor_scaler']['n_baseline_rows'])
    print('   mahal mean7 columns:', len(b['mahal_baseline']['mean7_columns']))
print('ALL BUNDLE ASSERTIONS PASSED')


In [ ]:
base = bundles['logreg']
SCALE_COLS = list(base['sensor_scaler']['columns'])
sc_scores = score_new(clean[SCALE_COLS], table['segment_id'], base)

healthy = clean.index[use & (clean.index.year == 2032)][0]
pre = clean.index[use & (clean.index < FAILURE_DATE)][-1]
shutdown_day = clean.index[state['is_shutdown'] & (clean.index.year == 2033)][0]
restart_day = clean.index[state['is_restart'] & (clean.index.year == 2033)][0]
first_op = clean.index[state['segment_id'].notna()][0]

print('healthy 2032 row      : P =', round(float(sc_scores.loc[healthy, 'probability']), 6),
      '| alarm =', bool(sc_scores.loc[healthy, 'alarm']))
print('pre-failure 2033 row  : P =', round(float(sc_scores.loc[pre, 'probability']), 6),
      '| alarm =', bool(sc_scores.loc[pre, 'alarm']))
print('shutdown day          : P =', sc_scores.loc[shutdown_day, 'probability'],
      '| alarm =', sc_scores.loc[shutdown_day, 'alarm'])
print('restart day           : P =', sc_scores.loc[restart_day, 'probability'],
      '| alarm =', sc_scores.loc[restart_day, 'alarm'])
print('first operating day   : P =', sc_scores.loc[first_op, 'probability'],
      '| alarm =', sc_scores.loc[first_op, 'alarm'])

assert not bool(sc_scores.loc[healthy, 'alarm'])
assert bool(sc_scores.loc[pre, 'alarm'])
assert pd.isna(sc_scores.loc[shutdown_day, 'probability']) and not bool(sc_scores.loc[shutdown_day, 'alarm'])
assert pd.isna(sc_scores.loc[restart_day, 'probability']) and not bool(sc_scores.loc[restart_day, 'alarm'])
assert pd.isna(sc_scores.loc[first_op, 'probability']) and not bool(sc_scores.loc[first_op, 'alarm'])

core_miss = clean.copy(); core_miss.loc[healthy, 'channel_098'] = np.nan
nc_miss = clean.copy(); nc_miss.loc[healthy, 'channel_005'] = np.nan
r_core = score_new(core_miss[SCALE_COLS], table['segment_id'], base).loc[healthy]
r_nc = score_new(nc_miss[SCALE_COLS], table['segment_id'], base).loc[healthy]
print()
print('missing core sensor     : P =', r_core['probability'], '| alarm =', r_core['alarm'])
print('missing non-core sensor : P =', round(float(r_nc['probability']), 6),
      '(reference', round(float(sc_scores.loc[healthy, 'probability']), 6), ')')
assert pd.isna(r_core['probability']) and not bool(r_core['alarm'])
assert abs(r_nc['probability'] - sc_scores.loc[healthy, 'probability']) < 1e-9

print('SCORING SAFETY CHECKS PASSED')


## Analysis step 9

This section uses synthetic values and anonymous channel identifiers.